In [1]:
import os
#
import tableauserverclient as TSC
#
import pandas as pd
import re
#
import time

In [2]:
tableau_server_url = "https://10ay.online.tableau.com/"
site_id = "lcomreporting"
personal_access_token_name = os.environ.get("Script_Token_Name")
personal_access_token_secret = os.environ.get("Script_Token_Secret")
#
batch_size = 3
sleep_time_sec = 300   # 5 minutes = 300 seconds`

In [3]:
# --- 1. Authenticate to Tableau Server ---
tableau_auth = TSC.PersonalAccessTokenAuth(
    token_name=personal_access_token_name,
    personal_access_token=personal_access_token_secret,
    site_id=site_id
)
server = TSC.Server(tableau_server_url, use_server_version=True)

In [4]:
with server.auth.sign_in(tableau_auth):
    # --- Step 1: Get all data sources ---
    all_datasources, _ = server.datasources.get()
    ds_dict = {ds.id: ds for ds in all_datasources}

    # --- Step 2: Get all extract refresh tasks ---
    all_tasks, _ = server.tasks.get()

    records = []
    for task in all_tasks:
        # Not all tasks have .target as expected, so guard with try/except
        try:
            target = task.target
            if hasattr(target, 'id') and target.id in ds_dict:
                ds = ds_dict[target.id]
                records.append({
                    "Datasource Name": ds.name,
                    "Datasource ID": ds.id,
                    "Tags": list(ds.tags),
                    "Extract Task ID": task.id
                })
        except Exception as e:
            continue

    # --- Step 3: Convert to DataFrame ---
    df = pd.DataFrame(records)


In [5]:
def extract_priority(tags):
        for tag in tags:
            match = re.match(r"Priority:(\d+)", tag)
            if match:
                return int(match.group(1))
        return 100

In [6]:
df["Priority"] = df["Tags"].apply(extract_priority)

In [7]:
df

,Datasource Name,Datasource ID,Tags,Extract Task ID,Priority
0,get_weekly_snapshot_data_district_level,03239258-ff1a-428d-bc16-43ef1d7b5a31,[Priority:2],211b9631-39d2-43f2-af20-e4d651709857,2
1,Customers Test2,7b7df199-c6a2-4b48-abf5-2d65b8a560fe,[],0b5ca6b3-e247-44ec-81e7-e1660e80b71d,100
2,Current Customer Metrics,46e71f76-24fc-4bdb-927a-ae85c93dcf01,[Priority:1],6c6b4fe6-af85-4ccc-bf96-d0c0caba70f9,1
3,Customers,5ee050f5-39b3-49ab-85f3-c8ee1e484390,[Priority:3],75926285-086b-4327-83dc-bedf4064fb23,3
4,Ultimate Parent Account Opportunities,b4e0ecf4-2629-44c1-9514-7796450535cb,[],737b565b-9a88-4c63-98c2-87e0a162d45a,100
5,Customers Test4,328e2014-aea2-4d16-bcfa-56331122b673,[],95b777a5-f652-4ffa-8a5f-93e8f2309efb,100
6,Customers Test5,5993d728-c8b4-49a8-96cf-a2efaa8cc6ad,[],a2d6d07e-5b52-46ef-9942-1dae65891dfb,100
7,get_monthly_snapshot_data_district_level,3195d74c-d655-4d07-80be-32373da7eec1,[Priority:2],9f753009-5375-491d-9697-5be309f50b3f,2
8,test_vw_monthly_snapshot_data_district_level (...,173e8ab2-f754-492d-b914-c891aad1abe4,[],15dfc256-6ca0-4b1b-82c1-10ea15a30aea,100
9,Product Categories,35ec3dc8-8cad-488e-92ef-44788fc822ff,[],ca94a3a5-d80b-4b61-9719-f8dd680ff95c,100


In [8]:
def refresh_now(extract_refresh_task_id):
    try:
        with server.auth.sign_in(tableau_auth):
        
            print(f"Attempting to run extract refresh task with ID: {extract_refresh_task_id}")

            extract_refresh_task = server.tasks.get_by_id(extract_refresh_task_id)  # Get the task object
            server.tasks.run(extract_refresh_task)  # Run the task

            print(f"Successfully triggered extract refresh task {extract_refresh_task_id}")            

    except TSC.ServerResponseError as e:
        print(f"Tableau Server Error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [9]:
df = df.sort_values(by="Priority", ascending=True).reset_index(drop=True)

In [11]:
for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i+batch_size]
    for _, row in batch.iterrows():
        task_id = row["Extract Task ID"]
        data_source_name = row["Datasource Name"]
        priority = row["Priority"]
        print(f"Triggering refresh for data source: {data_source_name} (Priority: {priority})")
        print("---")
        refresh_now(task_id)
    if i + batch_size < len(df):
        print(f"Waiting {sleep_time_sec} sec before next batch...")
        time.sleep(sleep_time_sec)  # 5 minutes = 300 seconds

Triggering refresh for data source: Current Customer Metrics (Priority: 1)
---
Attempting to run extract refresh task with ID: 6c6b4fe6-af85-4ccc-bf96-d0c0caba70f9
Successfully triggered extract refresh task 6c6b4fe6-af85-4ccc-bf96-d0c0caba70f9
Triggering refresh for data source: Current Licensing Metrics (Priority: 1)
---
Attempting to run extract refresh task with ID: 4c3cd08e-b7c2-4c3f-9fdc-58b4da6c402d
Successfully triggered extract refresh task 4c3cd08e-b7c2-4c3f-9fdc-58b4da6c402d
Triggering refresh for data source: Current Usage Metrics (Priority: 1)
---
Attempting to run extract refresh task with ID: 87a0e1cb-b0c9-4811-b773-2dceb3ae81e9
Successfully triggered extract refresh task 87a0e1cb-b0c9-4811-b773-2dceb3ae81e9
Waiting 300 sec before next batch...
Triggering refresh for data source: get_weekly_snapshot_data_district_level (Priority: 2)
---
Attempting to run extract refresh task with ID: 211b9631-39d2-43f2-af20-e4d651709857
Successfully triggered extract refresh task 211b963